# DEMO 14: Adding and Configuring a Genie Space

If dashboards operationalize **predefined** analytics (fixed reports, standard KPIs), **Genie** is how users **explore** analytics conversationally. Genie lets anyone ask questions in natural language and get answers as tables or visualizations — no SQL required from the end user.

| Dashboards | Genie |
| --- | --- |
| Fixed set of business questions | Open-ended questions |
| Author builds every visual | AI generates SQL + visuals on the fly |
| Viewers consume | Viewers explore |
| Great for operational reporting | Great for ad-hoc analysis |

### Two Types of Genie Spaces

| Type | How It’s Created | Use Case |
| --- | --- | --- |
| **Companion** | Automatically when you publish a dashboard | Let dashboard viewers ask follow-up questions |
| **Standalone** | Manually via Genie Spaces sidebar | Give business users a conversational interface without a fixed dashboard |

In this demo, we’ll:
1. See the companion Genie Space on our published dashboard
2. Create a standalone Genie Space from scratch
3. Configure the Knowledge Store (instructions, examples, metadata)
4. Test it with natural language questions

> **Prerequisite:** Run the setup cell below to create the source tables (with Unity Catalog comments that Genie will use). You should also have published the **Sales Performance Dashboard** from Demo 13.

---

In [0]:
%run ./demo_14_setup

## Part 1: The Companion Genie Space (Auto-Created)

When you publish a dashboard, Databricks automatically creates a companion Genie Space.

### How to access it:
1. Open your **published** Sales Performance Dashboard (not the draft)
2. Look for the **Ask Genie** button (top right, sparkle icon)
3. Click it — a conversational panel opens on the right side

### What it knows:
- All datasets defined in your dashboard
- Column names and types
- Any table/column comments from Unity Catalog

### Try asking it:
- *"What was total revenue last quarter?"*
- *"Which region had the most orders?"*
- *"Show me monthly revenue trend for Electronics"*
- *"Compare online vs retail store revenue"*

### What happens behind the scenes:
1. Genie interprets your natural language question
2. Generates an optimized SQL query against your dashboard datasets
3. Executes the query
4. Returns results as a table or auto-generated chart
5. You can ask follow-up questions (conversation context is maintained)

> **Power BI parallel:** Power BI has Q&A, but it requires the semantic model to be well-configured. Genie works similarly but uses a compound AI system that reasons across your data's full context (lineage, other queries, metadata).

---

## Part 2: Create a Standalone Genie Space

Standalone Genie Spaces are created independently of dashboards. Use these when you want to give business users conversational access to data without building a fixed dashboard first.

### Step 1: Navigate to Genie Spaces
1. Click **Genie** in the left sidebar
2. Click **New** (top right)

### Step 2: Configure the space
1. **Name:** `Sales Analytics Assistant`
2. **Description:** "Ask questions about sales performance, revenue, customers, and product trends."

### Step 3: Add tables
1. Click **Add tables** (or **Select tables**)
2. Browse Unity Catalog to find your tables:
   - `main.demo_dashboards_<your_username>.gold_sales`
   - `main.demo_dashboards_<your_username>.gold_customers`
3. Select both tables and click **Add**

### Step 4: Save
Click **Save** (or **Create**) — your Genie Space is now live!

### Test it:
Type a question in the chat input:
- *"How many orders were placed in 2024?"*
- *"What are the top 5 products by revenue?"*

Genie generates SQL and returns results. But notice: the answers might not perfectly understand your business terminology yet. That's where the Knowledge Store comes in.

---

## Part 3: The Knowledge Store — Teaching Genie Your Business

Genie's accuracy depends on how well it understands your data's **semantics**. The Knowledge Store is where you provide that context.

Think of Genie as a **new team member** — the more context you give them, the better they perform.

### Three components of the Knowledge Store:

| Component | What It Does | Example |
| --- | --- | --- |
| **Metadata annotations** | Connect business terms to columns | "revenue" = `net_revenue` column |
| **Example SQL queries** | Show Genie how your data is typically queried | "Top products" → GROUP BY product_name ORDER BY SUM(net_revenue) DESC |
| **Business-level guidance** | Explain rules, definitions, relationships | "Fiscal year starts April 1" |

---

## Step-by-Step: Add Metadata Annotations

Metadata annotations teach Genie what your columns mean in business terms.

### Step 1: Open your Genie Space settings
1. In your `Sales Analytics Assistant` space, click the **Settings** icon (gear) or **Edit** button
2. Navigate to the **Tables** or **Data** section

### Step 2: Add column descriptions
Click on a table (e.g., `gold_sales`) to see its columns. For each column, you can add:

| Column | Description to Add | Synonyms |
| --- | --- | --- |
| `net_revenue` | "Total revenue after discount" | revenue, sales, income |
| `product_category` | "Product grouping" | category, department |
| `customer_segment` | "Business size classification" | segment, customer type |
| `channel` | "How the order was placed" | sales channel, order source |
| `discount_pct` | "Percentage discount applied (0.0 to 0.20)" | discount |

### Step 3: Add value samples
For columns with specific values that users might reference differently:
- `region` column: sample values = `Northeast`, `Southeast`, `Midwest`, `West`, `Northwest`
- This helps Genie understand that "east coast" might mean "Northeast" or "Southeast"

> **Why this matters:** Without annotations, a user asking "What's our revenue?" might confuse Genie if there are multiple numeric columns. With a synonym of "revenue" on `net_revenue`, Genie knows exactly which column to use.

> **Note:** The setup script already added Unity Catalog **column comments** to both tables. Genie automatically reads these. The annotations above (synonyms, descriptions within the Genie Space) provide an additional layer of context on top of the UC comments.

---

## Step-by-Step: Add Example SQL Queries

Example queries teach Genie **how** your data is typically queried. They serve as templates for Genie's reasoning.

### Step 1: Navigate to Instructions
1. In your Genie Space settings, find the **Instructions** or **Example queries** section
2. Click **+ Add example query**

### Step 2: Add examples
For each example, provide a **natural language question** and the **SQL answer**:

**Example 1:**
- Question: *"What is total revenue by region?"*
- SQL:
```sql
SELECT region, ROUND(SUM(net_revenue), 2) AS total_revenue
FROM gold_sales
GROUP BY region
ORDER BY total_revenue DESC
```

**Example 2:**
- Question: *"Show me the monthly revenue trend"*
- SQL:
```sql
SELECT DATE_TRUNC('month', order_date) AS month,
       ROUND(SUM(net_revenue), 2) AS total_revenue
FROM gold_sales
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month
```

**Example 3:**
- Question: *"Who are the top 10 customers by lifetime value?"*
- SQL:
```sql
SELECT customer_name, segment, region,
       lifetime_orders, ROUND(lifetime_revenue, 2) AS lifetime_revenue
FROM gold_customers
ORDER BY lifetime_revenue DESC
LIMIT 10
```

**Example 4:**
- Question: *"What is the profit margin by product category?"*
- SQL:
```sql
SELECT product_category,
       ROUND(SUM(net_revenue), 2) AS revenue,
       ROUND(SUM(cost), 2) AS total_cost,
       ROUND((SUM(net_revenue) - SUM(cost)) / SUM(net_revenue) * 100, 1) AS profit_margin_pct
FROM gold_sales
GROUP BY product_category
ORDER BY profit_margin_pct DESC
```

**Example 5:**
- Question: *"Compare online and retail store performance"*
- SQL:
```sql
SELECT channel,
       COUNT(order_id) AS orders,
       ROUND(SUM(net_revenue), 2) AS revenue,
       ROUND(AVG(net_revenue), 2) AS avg_order_value
FROM gold_sales
WHERE channel IN ('Online', 'Retail Store')
GROUP BY channel
```

> **Best practice:** Add at least 5 example queries. For complex questions, having strong examples dramatically improves accuracy.

---

## Step-by-Step: Add Business-Level Guidance

Business guidance helps Genie understand your organizational logic — things that aren't obvious from the data alone.

### Step 1: Navigate to General Instructions
1. In your Genie Space settings, find the **General instructions** section
2. Add free-text instructions that explain your business rules

### Example instructions to add:

```
Business Rules:
- "Revenue" always refers to the net_revenue column (after discounts)
- Fiscal year runs January to December (calendar year)
- "Active customer" means a customer whose last_order_date is within the last 90 days
- When asked about "profit", calculate as: net_revenue - cost
- "Profit margin" is: (net_revenue - cost) / net_revenue * 100
- Round all monetary values to 2 decimal places
- Round all percentages to 1 decimal place
- Default date range is the year 2024 unless specified otherwise
- "Enterprise" and "Mid-Market" segments are considered B2B
- "SMB" and "Consumer" segments are considered B2C
```

### Step 2: Add SQL expressions (optional)
Some Genie Spaces allow you to define reusable SQL expressions:
- **Profit:** `SUM(net_revenue) - SUM(cost)`
- **Profit Margin %:** `ROUND((SUM(net_revenue) - SUM(cost)) / SUM(net_revenue) * 100, 1)`
- **B2B Revenue:** `SUM(CASE WHEN customer_segment IN ('Enterprise', 'Mid-Market') THEN net_revenue END)`

### Step 3: Add join relationships (if multiple tables)
Tell Genie how tables relate:
- `gold_sales.customer_segment` relates to `gold_customers.segment`
- `gold_sales.region` relates to `gold_customers.region`

> **Power BI parallel:** This is like documenting your semantic model — descriptions on measures, formatting rules, relationship definitions. The difference: in Power BI this lives in the model; in Genie it's explicitly authored guidance.

---

## Part 4: Test and Validate Genie

After configuring the Knowledge Store, test Genie to see the improvement.

### Test questions to try:
1. *"What's our total revenue?"* → Should use `net_revenue`, not `unit_price`
2. *"Show me profit margin by category"* → Should use the formula from your business rules
3. *"Who are our active customers?"* → Should filter by `last_order_date` within 90 days
4. *"How does B2B compare to B2C?"* → Should use your segment grouping definition
5. *"Monthly trend for the online channel"* → Should filter + time-group correctly

### Review the generated SQL:
- For each answer, Genie shows the SQL it generated
- Click to expand and verify it matches your expectations
- If something is wrong, use the feedback buttons

### Providing feedback:
For each Genie response, you can:
- **Yes** — confirms the response appears accurate (helps Genie learn)
- **Fix it** — flags as incorrect; select common issues or explain what's wrong
- **Request review** — flags for manual review by the space author

> **Important:** Genie is non-deterministic — identical questions might return slightly different SQL. Always verify results. Genie tells you WHAT happened, not WHY it happened.

---

## Part 5: Trusted Assets and Human Oversight

For mission-critical queries, you can create **Trusted Assets** — vetted, parameterized queries that Genie can reuse with confidence.

### What are Trusted Assets?
- SQL queries that have been **verified by an expert**
- Parameterized so they work for different filter values
- When Genie reuses the exact pattern, users can trust the output because the SQL was human-vetted

### Validation features:

| Feature | Purpose |
| --- | --- |
| **Trusted Assets** | Vetted queries for mission-critical logic |
| **Benchmarks** | Test suites comparing Genie’s SQL to ground-truth |
| **Generated SQL visibility** | Users can always see what SQL Genie wrote |
| **Feedback loop** | Yes / Fix it / Request review on every response |

### Best practices for Genie Spaces:
- Use well-annotated data (table and column comments in Unity Catalog)
- Add at least 5 example SQL queries
- Include business-specific instructions (“active customer” means X)
- Test with expected user questions before sharing
- Monitor quality and add feedback over time
- Think of Genie as a **new team member** who improves with clear guidance

---

---

## Summary

| Concept | Key Takeaway |
| --- | --- |
| **Companion Genie Space** | Auto-created when you publish a dashboard; "Ask Genie" button |
| **Standalone Genie Space** | Created independently; give users conversational access to any tables |
| **Knowledge Store** | Metadata + Examples + Business Rules = Genie accuracy |
| **Metadata annotations** | Column descriptions + synonyms + value samples |
| **Example SQL** | Teaches Genie how your data is typically queried (add 5+) |
| **Business guidance** | Rules, definitions, formulas that aren’t obvious from data alone |
| **Trusted Assets** | Vetted queries for mission-critical logic |
| **Feedback** | Yes / Fix it / Request review — Genie learns over time |

### Genie vs Dashboard — When to Use Each:

| Scenario | Best Tool |
| --- | --- |
| Monthly sales report everyone checks | **Dashboard** |
| "What happened last Tuesday in the West region?" | **Genie** |
| Executive KPI overview | **Dashboard** |
| Ad-hoc exploration by business analysts | **Genie** |
| Standardized operational metrics | **Dashboard** |
| Follow-up questions that a fixed dashboard can't answer | **Genie** |

**The best approach: use BOTH.** Publish a dashboard for standard reporting, and the companion Genie Space handles the ad-hoc questions that inevitably arise.